In [1]:
from torchvision import datasets, transforms
from torch import nn
import torch
from torch.utils.data import DataLoader
from matplotlib.pyplot import imshow, xlabel, ylabel, title
from math import floor

In [2]:
transform = transforms.Compose([transforms.RandomGrayscale(p= 0.2), transforms.RandomInvert(p = 0.5), transforms.RandomResizedCrop(224), transforms.ColorJitter(brightness= 0.2, contrast= 0.2, saturation= 0.2, hue = 0.2),transforms.RandomRotation(degrees= (-30,30)), transforms.RandomAdjustSharpness(sharpness_factor= 0.7, p= 0.3), transforms.ToTensor()])

train_data = datasets.Flowers102(root = "Flowers102", split = "train", download = True, transform = transforms.Compose([transforms.RandomResizedCrop(224),transforms.ToTensor()]))
test_data = datasets.Flowers102(root= "Flowers102", split = "test", download = True, transform= transforms.Compose([transforms.CenterCrop(224),transforms.ToTensor()]))

In [3]:
train_loader = DataLoader(dataset = train_data, batch_size = 64, drop_last= True)
test_loader = DataLoader(dataset = test_data, batch_size = 64, drop_last= True)

In [5]:
train_data[78][0].shape

torch.Size([3, 224, 224])

In [91]:
def resize(s, ker, str, pad):
    return floor((s + 2 * pad - ker) / str) + 1

resize(25, 3, 1, 1)

25

In [120]:
class Flower102(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels= 3, out_channels= 64, kernel_size= 7, padding = 3, stride= 2), ## 112
            nn.BatchNorm2d(num_features= 64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 3, stride= 2), ## 54
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size= 5, padding= 1, stride = 1), ## 52
            nn.BatchNorm2d(num_features= 128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=512, kernel_size= 3, padding= 1, stride = 1), ## 52
            nn.BatchNorm2d(num_features= 512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 3, stride= 2), ## 25
            nn.Conv2d(in_channels=512, out_channels = 1024, kernel_size= 3, padding= 1, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 1024),
            nn.ReLU(),
            nn.Conv2d(in_channels=1024, out_channels = 128, kernel_size= 1, padding= 0, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size= 3, padding= 1, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=1024, kernel_size= 1, padding= 0, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 1024),
            nn.ReLU(),
            nn.Conv2d(in_channels=1024, out_channels = 1024, kernel_size= 3, padding= 1, stride = 2), ## 12
            nn.BatchNorm2d(num_features= 1024),
            nn.ReLU(),
            nn.Conv2d(in_channels=1024, out_channels=256, kernel_size= 1, padding= 0, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 256),
            nn.ReLU(),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size= 3, padding= 1, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 256),
            nn.ReLU(),
            nn.Conv2d(in_channels=256, out_channels=1024, kernel_size= 1, padding= 0, stride = 1), ## 25
            nn.BatchNorm2d(num_features= 1024),
            nn.ReLU(),
            nn.Conv2d(in_channels=1024, out_channels=2048, kernel_size= 3, padding= 1, stride = 2), ## 13
            nn.BatchNorm2d(num_features= 2048),
            nn.ReLU(),
            nn.Conv2d(in_channels=2048, out_channels=512, kernel_size= 1, padding= 0, stride = 1), ## 13
            nn.BatchNorm2d(num_features= 512),
            nn.ReLU(),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size= 3, padding= 1, stride = 1), ## 13
            nn.BatchNorm2d(num_features= 512),
            nn.ReLU(),
            nn.Conv2d(in_channels=512, out_channels=2048, kernel_size= 1, padding= 0, stride = 1), ## 13
            nn.BatchNorm2d(num_features= 2048),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(output_size= (1,1))
        )

        self.dense_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 500),
            nn.ReLU(),
            nn.Linear(500,250),
            nn.ReLU(),
            nn.Linear(250,102)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = self.dense_layers(x)

        return x
    
    def get_pred(self,x):
        self.eval()
        with torch.inference_mode():
            x = self.forward(x)

        return torch.argmax(torch.softmax(x, dim= 1), dim = 1)

In [121]:
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ExponentialLR

In [127]:
torch.manual_seed(2009)

model = Flower102().to("cuda")


loss_fn = nn.CrossEntropyLoss()

epoch_num = 100

optimizer = torch.optim.Adam(params = model.parameters(), lr = 3e-4)
scheduler = ExponentialLR(optimizer, gamma=0.9995) 


In [130]:
accum_steps = 4
scaler = GradScaler(device= "cuda")


for epoch in range(epoch_num*10000):
    n_r = 0
    i = 0

    optimizer.zero_grad()
    for batch_num, (X,y) in enumerate(train_loader):
        model.train()
        X, y = X.to("cuda"), y.to("cuda")
        
        i += y.size(0)
        with autocast("cuda"):
            logits = model(X)
            loss = loss_fn(logits, y) / accum_steps

            n_r += (model.get_pred(X) == y).sum().item()
        
        scaler.scale(loss).backward()
        if (batch_num + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            
    scheduler.step()
    print(f"Epoch : {epoch} Accuracy : {n_r / i}")

if (batch_num + 1) % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()



Epoch : 0 Accuracy : 0.21354166666666666
Epoch : 1 Accuracy : 0.21145833333333333
Epoch : 2 Accuracy : 0.21354166666666666
Epoch : 3 Accuracy : 0.19791666666666666
Epoch : 4 Accuracy : 0.20833333333333334
Epoch : 5 Accuracy : 0.215625
Epoch : 6 Accuracy : 0.209375
Epoch : 7 Accuracy : 0.1875
Epoch : 8 Accuracy : 0.19375
Epoch : 9 Accuracy : 0.19270833333333334
Epoch : 10 Accuracy : 0.20208333333333334
Epoch : 11 Accuracy : 0.21041666666666667
Epoch : 12 Accuracy : 0.20520833333333333
Epoch : 13 Accuracy : 0.19270833333333334
Epoch : 14 Accuracy : 0.21458333333333332
Epoch : 15 Accuracy : 0.20520833333333333
Epoch : 16 Accuracy : 0.18125
Epoch : 17 Accuracy : 0.19479166666666667
Epoch : 18 Accuracy : 0.19375
Epoch : 19 Accuracy : 0.19895833333333332
Epoch : 20 Accuracy : 0.215625
Epoch : 21 Accuracy : 0.16458333333333333
Epoch : 22 Accuracy : 0.18645833333333334
Epoch : 23 Accuracy : 0.209375
Epoch : 24 Accuracy : 0.18125
Epoch : 25 Accuracy : 0.19479166666666667
Epoch : 26 Accuracy : 0

KeyboardInterrupt: 

In [131]:
for batch_num, (X,y) in enumerate(train_loader):
        X, y = X.to("cuda"), y.to("cuda")
        
        i += y.size(0)

        n_r += (model.get_pred(X) == y).sum().item()
        
            
print(f"Accuracy : {n_r / i}")

Accuracy : 0.20474137931034483
